In [1]:
import site
import sys
import subprocess
import os

# 1. Registrar la ruta de paquetes del usuario en el sistema
user_site = '/headless/.local/lib/python3.12/site-packages'
if os.path.exists(user_site):
    site.addsitedir(user_site)

# 2. Intentar importar; si falla, instalarlo automáticamente
try:
    from PyLTSpice import RawRead
    print('¡PyLTSpice ya estaba instalado y listo para usar!')
except ImportError:
    print('PyLTSpice no fue encontrado. Instalando automáticamente...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'PyLTSpice'])
    if os.path.exists(user_site):
        site.addsitedir(user_site)
    from PyLTSpice import RawRead
    print('¡PyLTSpice se instaló e importó correctamente!')

¡PyLTSpice ya estaba instalado y listo para usar!


In [2]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import scipy.stats as stats
import os

# ==============================================================================
# 1. CARGAR Y EXTRAER LOS DATOS
# ==============================================================================
ruta_archivo = "/headless/.xschem/simulations/datos_mc.txt"

if not os.path.exists(ruta_archivo):
    print(f"⚠️ Archivo no encontrado en: {ruta_archivo}")
else:
    with open(ruta_archivo, 'r') as f:
        texto = f.read()

    # Función auxiliar para extraer números con Regex
    def extraer(variable, texto):
        patron = rf"{variable}\s*=\s*([\d\.\-eE\+]+)"
        encontrados = re.findall(patron, texto)
        
        resultados = []
        for x in encontrados:
            try:
                resultados.append(float(x))
            except ValueError:
                resultados.append(np.nan) 
                
        return resultados

    # Diccionario con TODOS los parámetros medidos en Ngspice
    datos_extraidos = {
        'Dropout (mV)': extraer('dropout_mV', texto),
        'Margen Fase (Deg)': extraer('margen_fase', texto),
        'UGB (Hz)': extraer('ugb', texto),
        'Overshoot (mV)': extraer('overshoot_mV', texto),
        'Undershoot (mV)': extraer('undershoot_mV', texto),
        'Line Reg (mV)': extraer('line_regulation_mv', texto),
        'Load Reg (mV)': extraer('load_regulation_mV', texto),
        'PSRR 1k (dB)': extraer('psrr_1k', texto),
        'PSRR 10k (dB)': extraer('psrr_10k', texto),
        'Voltaje de salida en line (V)': extraer('salida_line', texto),
        'Voltaje de salida en tran (V)': extraer('salida_tran', texto),
    }

    # Filtramos solo las variables presentes (compatible con Esquemático y PEX)
    datos_extraidos = {k: v for k, v in datos_extraidos.items() if len(v) > 0}

    # Creamos el DataFrame
    df_mc = pd.DataFrame.from_dict(datos_extraidos, orient='index').T
    iteraciones_originales = len(df_mc)

    # ==============================================================================
    # 1.5. FILTRO DE CONVERGENCIA (LIMPIEZA DE DATOS)
    # ==============================================================================
    # a. Eliminar iteraciones con datos en blanco (NaN)
    """
    df_mc = df_mc.dropna()
# Solo filtramos basuras matemáticas (valores fuera de los rieles de alimentación)
    if 'Voltaje de salida en tran (V)' in df_mc.columns:
        df_mc = df_mc[(df_mc['Voltaje de salida en tran (V)'] >= 0.0) & (df_mc['Voltaje de salida en tran (V)'] <= 1.8)]
    # b. Filtrar divergencias en el voltaje de salida transitorio


    # c. Filtrar divergencias en el voltaje de salida de línea
    if 'Voltaje de salida en line (V)' in df_mc.columns:
        df_mc = df_mc[(df_mc['Voltaje de salida en line (V)'] > 0) & (df_mc['Voltaje de salida en line (V)'] < 1.8)]

    # d. Filtrar anomalías transitorias extremas
    if 'Undershoot (mV)' in df_mc.columns:
        df_mc = df_mc[df_mc['Undershoot (mV)'] < 200]
        
    if 'Dropout (mV)' in df_mc.columns:
        df_mc = df_mc[df_mc['Dropout (mV)'] < 200]

    # e. Filtrar basuras matemáticas en las Regulaciones
    if 'Load Reg (mV)' in df_mc.columns:
        df_mc = df_mc[(df_mc['Load Reg (mV)'] > -50) & (df_mc['Load Reg (mV)'] < 50)]
        
    if 'Line Reg (mV)' in df_mc.columns:
        df_mc = df_mc[(df_mc['Line Reg (mV)'] > -50) & (df_mc['Line Reg (mV)'] < 50)]
        
    print(f"✓ Limpieza completada: Se retuvieron {len(df_mc)} iteraciones válidas de {iteraciones_originales} totales.")
"""
    # Mostramos las primeras 10 corridas limpias
    try:
        display(df_mc.head(10)) 
    except NameError:
        print(df_mc.head(10))

# ==============================================================================
# 2. FUNCIÓN PARA GRAFICAR HISTOGRAMAS Y CAMPANA DE GAUSS
# ==============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

def graficar_gauss(df, columna, carpeta_salida='graficos_salida'):
    datos = df[columna].dropna() 
    if len(datos) < 2: return
    
    # 1. Crear la carpeta si no existe
    os.makedirs(carpeta_salida, exist_ok=True)
    
    mu = np.mean(datos)            
    sigma = np.std(datos, ddof=1)  
    
    plt.figure(figsize=(8, 4))
    
    count, bins, ignored = plt.hist(datos, bins=15, density=True, alpha=0.6, color='#2c7bb6', edgecolor='black')
    
    xmin, xmax = plt.xlim()
    x = np.linspace(xmin, xmax, 100)
    p = stats.norm.pdf(x, mu, sigma)
    plt.plot(x, p, 'k', linewidth=2, label='Distribución Normal')
    
    plt.axvline(mu, color='red', linestyle='dashed', linewidth=1.5, label=fr'Media ($\mu$): {mu:.4f}')
    plt.axvline(mu - 3*sigma, color='green', linestyle='dotted', linewidth=1.5, label=fr'-3$\sigma$: {mu-3*sigma:.4f}')
    plt.axvline(mu + 3*sigma, color='green', linestyle='dotted', linewidth=1.5, label=fr'+3$\sigma$: {mu+3*sigma:.4f}')
    
    plt.title(f'Análisis Monte Carlo: {columna} (N={len(datos)})', fontsize=12, fontweight='bold')
    plt.xlabel(columna, fontsize=11)
    plt.ylabel('Densidad de Probabilidad', fontsize=11)
    plt.legend(loc='best')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    # 2. Guardar la figura en la carpeta
    ruta_archivo = os.path.join(carpeta_salida, f'distribucion_{columna}.png')
    plt.savefig(ruta_archivo, dpi=300, bbox_inches='tight')
    
    # 3. Cerrar la figura para liberar memoria en el bucle
    plt.close()

# ==============================================================================
# 3. GENERAR EL REPORTE VISUAL
# ==============================================================================
if 'df_mc' in locals() and not df_mc.empty:
    # Puedes cambiar el nombre de la carpeta aquí
    nombre_carpeta = 'Resultados_Montecarlo'
    
    for columna in df_mc.columns:
        graficar_gauss(df_mc, columna, carpeta_salida=nombre_carpeta)

<>:117: SyntaxWarning: invalid escape sequence '\m'
<>:118: SyntaxWarning: invalid escape sequence '\s'
<>:119: SyntaxWarning: invalid escape sequence '\s'
<>:117: SyntaxWarning: invalid escape sequence '\m'
<>:118: SyntaxWarning: invalid escape sequence '\s'
<>:119: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_7670/3669540200.py:117: SyntaxWarning: invalid escape sequence '\m'
  plt.axvline(mu, color='red', linestyle='dashed', linewidth=1.5, label=f'Media ($\mu$): {mu:.4f}')
/tmp/ipykernel_7670/3669540200.py:118: SyntaxWarning: invalid escape sequence '\s'
  plt.axvline(mu - 3*sigma, color='green', linestyle='dotted', linewidth=1.5, label=f'-3$\sigma$: {mu-3*sigma:.4f}')
/tmp/ipykernel_7670/3669540200.py:119: SyntaxWarning: invalid escape sequence '\s'
  plt.axvline(mu + 3*sigma, color='green', linestyle='dotted', linewidth=1.5, label=f'+3$\sigma$: {mu+3*sigma:.4f}')


,Dropout (mV),Margen Fase (Deg),UGB (Hz),Overshoot (mV),Undershoot (mV),Line Reg (mV),Load Reg (mV),PSRR 1k (dB),PSRR 10k (dB),Voltaje de salida en line (V),Voltaje de salida en tran (V)
0,51.2321,NaN,0.0,2.099,3.128,0.489,0.737,69.6251,61.4678,1.18802,1.19880
1,50.1638,NaN,0.0,2.018,3.006,0.443,0.727,62.0364,59.7113,1.18899,1.18868
2,50.9049,NaN,0.0,2.048,3.051,0.428,0.763,62.0577,59.7492,1.20210,1.18967
3,51.9488,NaN,0.0,2.168,3.228,0.462,0.740,54.1371,53.8450,1.20154,1.21131
4,52.7913,NaN,0.0,2.119,3.163,0.435,0.729,53.6939,53.4532,1.18715,1.19999
5,51.9917,NaN,0.0,2.069,3.064,0.487,0.736,82.1306,62.7465,1.20302,1.20070
6,51.7263,NaN,0.0,2.072,3.086,0.462,0.745,80.6064,62.6300,1.21196,1.19962
7,53.4141,NaN,0.0,2.135,3.170,0.439,0.738,63.1565,60.2580,1.19252,1.21080
8,50.5219,NaN,0.0,2.182,3.272,0.467,0.713,57.6927,56.9041,1.20782,1.19912
9,53.3594,NaN,0.0,2.081,3.109,0.441,0.739,81.5437,62.8469,1.20648,1.19502


/usr/local/lib/python3.12/dist-packages/scipy/stats/_distn_infrastructure.py:2076: RuntimeWarning: divide by zero encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
